# Source windows, gaps and breaks

## Overview
Each source covers its own stretch of time, with gaps where the machine or the logging stopped. On top of that, some process changes - a new version of a routine, a new gate on DS3 values, a coupling flipped round - can shift a series with no physics behind them. We need to know where all of these are, so that later work doesn't read a gap or a process change as drift.

**Question.** When does each source have data, where are the gaps, and where did the process or the machine's state change in a way that could look like physics?

**Method.** We take the first and last timestamp of each source (section 1) and look for gaps of 3 days or more (section 2). Then we look for breaks in DS3 (section 3) and in the routines (section 4), and for DS2 changes no tracked run made, including the ones that end each outage (section 5). Last, we check every entry in `qpu_drift.periods` against what we found (section 6). Results are in [Phase 01, A4](../../docs/phases/01_eda.md#a4-study-periods-gaps-and-breaks).

In [1]:
import pandas as pd

from qpu_drift.derived.run_map import load_run_entities, load_run_map, load_run_results
from qpu_drift.sources.benchmarking import load_kpi_l2, load_runs_l2
from qpu_drift.sources.fridge import load_fridge
from qpu_drift.sources.hardware_models import load_index
from qpu_drift.sources.measurements import load_measurements

# Run map and the four sources
runs = load_run_map()
ds1 = load_measurements()
ds2 = load_index()
ds3 = load_runs_l2().sort_values("created_at")
kpi = load_kpi_l2()
ds4 = load_fridge()

## 1. All four sources overlap from 2025-08-01 to 2026-08-18

First, when each source starts and stops.

In [2]:
# First and last timestamp of each source
pd.DataFrame(
    {
        "output_info": [runs["start"].min(), runs["start"].max()],
        "DS1": [ds1["time_start"].min(), ds1["time_start"].max()],
        "DS2": [ds2["timestamp"].min(), ds2["timestamp"].max()],
        "DS3": [ds3["created_at"].min(), ds3["created_at"].max()],
        "DS4": [ds4.index.min().tz_localize(None), ds4.index.max().tz_localize(None)],
    },
    index=["first", "last"],
).T

,first,last
output_info,2024-03-21 17:03:19.000000,2026-08-18 09:03:54.000000
DS1,2024-09-18 11:53:21.161967,2026-08-18 09:12:51.648211
DS2,2025-01-01 08:00:13.000000,2026-08-18 09:13:31.000000
DS3,2024-05-24 20:16:42.918570,2026-09-14 22:01:36.723743
DS4,2025-08-01 12:00:00.000000,2026-09-03 12:00:00.000000


`output_info` starts first and DS3 runs on longest. DS1 starts at the restore, DS2 at the start of 2025 and DS4 in 2025-08.

| Source | First | Last |
|---|---|---|
| `output_info` | 2024-03-21 | 2026-08-18 |
| DS1 | 2024-09-18 (the restore) | 2026-08-18 |
| DS2 | 2025-01-01 | 2026-08-18 |
| DS3 | 2024-05-24 | 2026-09-14 |
| DS4 | 2025-08-01 | 2026-09-03 |

So all four overlap from 2025-08-01 to 2026-08-18, and DS1 to DS3 from 2025-01-01. There's no fridge record before 2025-08.

## 2. Three outages stop every routine

We look for stretches of 3 days or more with no non-empty run of any routine; from 2025 on, we call these outages. Then we check the DS4 readings for gaps, run the same check on each source's own rows (DS1 `time_start`, DS2 snapshots, DS3 runs) and see what each source still has inside the outages.

In [3]:
# Days with no non-empty run of any routine, in stretches of 3 days or more
active = pd.DatetimeIndex(
    runs.loc[runs["n_result_ids"] > 0, "start"].dt.normalize().unique()
).sort_values()
off = pd.Series(pd.date_range(active.min(), active.max(), freq="D").difference(active))
stretch = (off.diff() != pd.Timedelta("1D")).cumsum()
gaps = off.groupby(stretch).agg(["min", "max", "size"])
display(gaps[gaps["size"] >= 3].reset_index(drop=True))

# 2024-05-21 to 10-14: routines with any run, and the DS3 runs in the same period
mid = runs[(runs["start"] >= "2024-05-20") & (runs["start"] < "2024-10-15")]
display(mid.groupby("routine")["start"].agg(["size", "min", "max"]))
ds3_mid = ds3[(ds3["created_at"] >= "2024-05-20") & (ds3["created_at"] < "2024-10-15")]
print(f"DS3 runs in the same period: {len(ds3_mid)}, last {ds3_mid['created_at'].max()}")

# DS4: steps between readings over 6 h
step = ds4.index.to_series().diff()
step[step > pd.Timedelta("6h")].rename("gap ending here")

,min,max,size
0,2024-03-22,2024-04-10,20
1,2024-05-21,2024-09-17,120
2,2024-09-21,2024-10-14,24
3,2025-02-05,2025-02-11,7
4,2025-03-10,2025-03-17,8
5,2026-03-12,2026-03-23,12


,size,min,max
routine,,,
calibrate_1q,4,2024-05-20 05:30:07,2024-09-18 16:24:41
mixer_calibration,22,2024-09-18 12:29:27,2024-09-20 23:40:15


DS3 runs in the same period: 75, last 2024-08-17 02:50:01.260564


timestamp
2026-03-17 18:00:00+00:00    0 days 18:00:00
2026-07-02 18:00:00+00:00   21 days 00:00:00
2026-07-08 18:00:00+00:00    1 days 12:00:00
Name: gap ending here, dtype: timedelta64[ms]

Six stretches have no non-empty run. Three are in 2024: 2024-03-22 to 04-10 (20 days), right after the first `output_info` file, then 2024-05-21 to 09-17 (120 days) and 2024-09-21 to 10-14 (24 days). From 2024-05-20 to 10-14 the only files are 4 `calibrate_1q` runs and 22 mixer runs, while DS3 keeps going until 2024-08-17 with 75 runs. The other three are the outages: 2025-02-05 to 02-11, 2025-03-10 to 03-17 and 2026-03-12 to 03-23.

DS4 has three steps over 6 h, ending on 2026-03-17 (18 h), 2026-07-02 (21 days) and 2026-07-08 (36 h). Next, how often each step between DS4 readings occurs.

In [4]:
# How often each step between DS4 readings occurs
step.value_counts().head().rename("DS4 steps between readings")

timestamp
0 days 06:00:00     1499
0 days 18:00:00        1
21 days 00:00:00       1
1 days 12:00:00        1
Name: DS4 steps between readings, dtype: int64

The other 1,499 steps are all 6 h, so DS4 reads every 6 h and those three are its only gaps.

Next, what each source still has inside the outages, and each source's own gaps.

In [5]:
def off_stretches(times: pd.Series) -> pd.DataFrame:
    """Stretches of 3 or more days with no rows, between a source's first and last day."""
    days = pd.DatetimeIndex(pd.to_datetime(times).dt.normalize().unique()).sort_values()
    off = pd.Series(pd.date_range(days.min(), days.max(), freq="D").difference(days))
    g = off.groupby((off.diff() != pd.Timedelta("1D")).cumsum()).agg(["min", "max", "size"])
    return g[g["size"] >= 3].reset_index(drop=True)


# Rows of each source
sources = {
    "DS1": ds1["time_start"],
    "DS2": pd.to_datetime(ds2["timestamp"]),
    "DS3": ds3["created_at"],
}

# Outages from 2025: stretches of 3 days or more with no run
outage_rows = gaps[(gaps["size"] >= 3) & (gaps["min"] >= pd.Timestamp("2025-01-01"))]

# Rows per day that each source has inside each outage
inside = pd.DataFrame(
    {
        name: [
            t[(t >= g["min"]) & (t < g["max"] + pd.Timedelta("1D"))]
            .dt.date.value_counts()
            .sort_index()
            .to_dict()
            for _, g in outage_rows.iterrows()
        ]
        for name, t in sources.items()
    },
    index=[f"{g['min'].date()} to {g['max'].date()}" for _, g in outage_rows.iterrows()],
).rename_axis("rows per day inside outage")
display(inside)

# Runs inside each outage and their result ids
for _, g in outage_rows.iterrows():
    r = runs[(runs["start"] >= g["min"]) & (runs["start"] < g["max"] + pd.Timedelta("1D"))]
    print(f"runs in {g['min'].date()} to {g['max'].date()}: {len(r)}")
    if len(r):
        print(r[["start", "routine", "n_result_ids"]].to_string(index=False))

# Each source's own stretches with no rows
pd.concat({name: off_stretches(t) for name, t in sources.items()}).droplevel(1)

,DS1,DS2,DS3
rows per day inside outage,,,
2025-02-05 to 2025-02-11,{},"{2025-02-05: 8, 2025-02-06: 3}",{}
2025-03-10 to 2025-03-17,{},{},{}
2026-03-12 to 2026-03-23,{},{2026-03-23: 1},{}


runs in 2025-02-05 to 2025-02-11: 4
              start      routine  n_result_ids
2025-02-05 08:00:27 calibrate_1q             0
2025-02-05 08:01:13 calibrate_2q             0
2025-02-06 08:00:28 calibrate_1q             0
2025-02-06 08:01:14 calibrate_2q             0
runs in 2025-03-10 to 2025-03-17: 0
runs in 2026-03-12 to 2026-03-23: 0


,min,max,size
DS1,2024-09-26,2024-10-14,19
DS1,2025-02-05,2025-02-11,7
DS1,2025-03-10,2025-03-17,8
DS1,2026-03-12,2026-03-23,12
DS2,2025-02-07,2025-02-11,5
DS2,2025-03-10,2025-03-17,8
DS2,2026-03-12,2026-03-22,11
DS3,2024-07-19,2024-07-24,6
DS3,2024-08-18,2024-10-14,58
DS3,2024-10-18,2024-10-20,3


DS1 and DS3 have no rows inside the outages, and DS2 only a few: 8 on 2025-02-05, 3 on 02-06 and 1 on 2026-03-23. The first outage has four runs, a `calibrate_1q` and a `calibrate_2q` on each of 02-05 and 02-06, all with no result ids. The other two have none.

Each source's own gaps are mostly the outages. DS1 adds 2024-09-26 to 10-14 (19 days), and DS2's are shorter where it has rows: 2025-02-07 to 02-11, 2025-03-10 to 03-17 and 2026-03-12 to 03-22. DS3's outage gaps are 2025-02-05 to 02-11, 2025-03-09 to 03-20 and 2026-03-12 to 03-23. It adds 2024-08-18 to 10-14 (58 days) and ten short gaps of 3 to 6 days, starting 2024-07-19, 2024-10-18, 2024-10-25, 2024-12-09, 2025-04-17, 2026-01-20, 2026-05-22, 2026-07-02, 2026-08-23 and 2026-09-05.

## 3. DS3 flips couplings twice and changes its gate three times

Two things in DS3 can change with no physics behind them: which couplings it reports (and which way round), and which gate decides whether a value is kept.

In [6]:
# Topology changes: couplings removed and added at each change
edges = ds3["topology"].map(lambda t: frozenset(t))
change = ds3[edges != edges.shift()]
for prev, (_, row) in zip([None, *change["topology"].iloc[:-1]], change.iterrows()):
    if prev is not None:
        print(
            row["created_at"],
            "removed",
            sorted(set(prev) - set(row["topology"])),
            "added",
            sorted(set(row["topology"]) - set(prev)),
        )

# First run of each gating era, then the first error, R2 and openpulse values
print(ds3.groupby("gating_regime")["created_at"].min())
print("first *_error:", kpi.loc[kpi["error"].notna(), "created_at"].min())
print("first R²:", kpi.loc[kpi["r2"].notna(), "created_at"].min())
print("first openpulse:", ds3.loc[ds3["has_openpulse"], "created_at"].min())

# Share of kpi_l2 rows with an R2 in each era
kpi.merge(ds3[["created_at", "gating_regime"]], on="created_at").groupby("gating_regime")[
    "r2"
].agg(rows="size", with_r2=lambda s: round(s.notna().mean(), 2))

2024-05-28 20:10:56.372690 removed ['2-1', '25-17'] added ['1-2', '17-25']
2026-03-24 23:48:40.391568 removed ['1-10', '1-2'] added ['10-1', '2-1']
gating_regime
era0_ungated    2024-05-24 20:16:42.918570
era1_filter     2024-10-15 19:39:16.280236
era2_ungated    2025-07-07 23:10:10.082632
era3_postproc   2025-08-22 23:12:56.049756
Name: created_at, dtype: datetime64[us]
first *_error: 2025-07-07 23:10:10.082632
first R²: 2024-10-15 19:39:16.280236
first openpulse: 2026-05-21 23:00:56.457438


,rows,with_r2
gating_regime,,
era0_ungated,12000,0.00
era1_filter,35680,0.64
era2_ungated,6400,0.77
era3_postproc,50720,0.77


Couplings flip twice: on 2024-05-28 `2-1` and `25-17` become `1-2` and `17-25`, and on 2026-03-24 `1-10` and `1-2` become `10-1` and `2-1`.

Era 0 starts on 2024-05-24, era 1 on 2024-10-15, era 2 on 2025-07-07 and era 3 on 2025-08-22. The gate goes from none, to $R^2$ only, back to none, then to $R^2$ and range ([datasheet 3, section 3](../../docs/datasheets/03_benchmarking_files.md#3-provenance-and-collection)). `*_error` starts with era 2, and `openpulse_features` on 2026-05-21.

$R^2$ is missing from all of era 0, and is on 64% of era 1 rows and 77% of eras 2 and 3. The served payload only has it from era 2, so era 1's comes from `output_info` (same section).

## 4. `calibrate_1q` switched to new code in 2025-02, with eight days back on the old code in 2025-03

Commit 69bc440 in `oqc-calibration-routines` changed `calibrate_1q` in three ways: `drag_limit` went from $4\times10^{-9}$ to $9\times10^{-9}$, the DRAG ladder from $N=1,2,4$ to $N=2,3,4$, and the FineFrequency delays from 0.1 and 0.25 µs to 0.1, 0.25 and 0.5 µs. To see which version ran when, we count the FineFrequency results per qubit in each run (the median, i.e. the number of delays). Along the way we check the start times and the runs that only measure some qubits.

In [7]:
# FineFrequency results per qubit (delays), median per run
ids = load_run_results()
ff = ids[(ids["routine"] == "calibrate_1q") & ids["valid"] & (ids["step"] == "freq")]
delays = ff.groupby(["start", "key"])["result_id"].nunique().groupby("start").median()

# Stretches of runs with the same delay count
segment = (delays != delays.shift()).cumsum()
display(
    delays.groupby(segment)
    .agg(delays="first", first=lambda s: s.index.min(), last=lambda s: s.index.max(), runs="size")
    .reset_index(drop=True)
)

# Most common start times before and after 2025-03-20
c1q = runs[(runs["routine"] == "calibrate_1q") & runs["start"].between("2025-01-01", "2025-06-01")]
print("most common calibrate_1q start times, 2025-01 to 2025-05:")
print(
    c1q.groupby([c1q["start"] >= "2025-03-20", "window"])["start"]
    .agg(lambda s: s.dt.strftime("%H:%M").value_counts().head(2).to_dict())
    .rename_axis(["from 2025-03-20", "window"])
    .to_string()
)

# Qubits measured per run, and where a run measures under half the usual
ent = load_run_entities(routine="calibrate_1q")
measured = (
    ent[(ent["param"] == "freq") & ent["measured"].fillna(False)]
    .groupby("start")["entity"]
    .nunique()
)
reduced = measured < measured.median() / 2  # runs measuring under half the usual qubits
print("qubits measured per run:", measured.value_counts().to_dict())

# First reduced run, then first full run after
measured[reduced != reduced.shift(fill_value=False)]

,delays,first,last,runs
0,2.0,2024-09-18 15:13:54,2025-02-17 14:18:25,221
1,3.0,2025-02-17 16:11:02,2025-03-20 08:00:20,40
2,2.0,2025-03-20 12:00:58,2025-03-28 08:45:22,18
3,3.0,2025-03-28 19:44:15,2026-03-08 19:42:41,678
4,2.0,2026-03-09 08:45:16,2026-03-09 08:45:16,1
5,3.0,2026-03-09 19:42:45,2026-03-09 19:42:45,1
6,2.0,2026-03-10 08:45:17,2026-03-10 08:45:17,1
7,3.0,2026-03-10 19:42:52,2026-03-10 19:42:52,1
8,2.0,2026-03-11 08:45:16,2026-03-11 08:45:16,1
9,1.0,2026-03-11 19:43:57,2026-03-11 19:43:57,1


most common calibrate_1q start times, 2025-01 to 2025-05:
from 2025-03-20  window 
False            evening    {'19:52': 29, '19:45': 27}
                 morning     {'08:00': 63, '08:44': 1}
                 other        {'13:41': 1, '14:18': 1}
True             evening    {'19:44': 46, '19:43': 22}
                 morning     {'08:45': 72, '08:00': 1}
                 other        {'12:00': 1, '12:40': 1}
qubits measured per run: {30: 1021, 29: 232, 14: 14, 32: 4, 28: 2}


start
2025-04-24 08:45:21    14
2025-04-25 12:13:42    30
2026-03-10 08:45:17    14
2026-03-24 14:45:52    30
2026-05-18 08:45:17    14
2026-05-18 19:52:10    30
2026-07-03 08:45:41    14
2026-07-05 21:54:09    30
Name: entity, dtype: int64

The delay count shows which version ran: 2 delays up to 2025-02-17 14:18 (221 runs), 3 from 16:11 that day (40 runs), 2 again from 2025-03-20 12:00 to 03-28 08:45 (18 runs), then 3 from 2025-03-28 19:44 (678 runs, to 2026-03-08). After that it stays at 3, apart from single runs with 1, 2 or 2.5 delays on 2026-03-09 to 03-11, 2026-07-02 and 07-03, and 2026-08-11. The start times move on 2025-03-20 too, from mostly 08:00 and 19:52 or 19:45 to 08:45 and 19:44 or 19:43.

14 `calibrate_1q` runs measure only 14 qubits, in four stretches starting 2025-04-24, 2026-03-10, 2026-05-18 and 2026-07-03 (all at 08:45). Full runs come back on 2025-04-25 12:13, 2026-03-24 14:45, 2026-05-18 19:52 and 2026-07-05 21:54. These aren't breaks: the `measured` flag drops the qubits they skip.

Next, the commit dates, and the lowest DRAG $N$ warned in each run against its delay count. If the old ladder ran with the old delays, $N=1$ warnings should only turn up in 2-delay runs.

In [8]:
import re
import subprocess
from pathlib import Path

from qpu_drift.paths import OUTPUT_INFO
from qpu_drift.sources.output_info import iter_files

ROUTINES = Path.home() / "git" / "oqc-calibration-routines"


def git(*args: str) -> str:
    """Run git in the local oqc-calibration-routines clone."""
    return subprocess.run(
        ["git", "-C", ROUTINES, *args], capture_output=True, text=True, check=True
    ).stdout


# The two commits, then the lines 69bc440 changed
for sha in ("69bc440", "48b8810"):
    print(git("log", "-1", "--format=%h | author %ai | committer %ci | %s", sha), end="")
print(
    "\n".join(
        l
        for l in git("show", "69bc440", "-U0", "--format=").splitlines()
        if l[:1] in "+-" and l[:3] not in ("+++", "---")
    )
)

# Lowest DRAG N warned in each calibrate_1q run
drag_n = pd.Series(
    {
        start: min(
            (
                int(m[1])
                for w in map(str, oi.get("warnings") or [])
                if (m := re.search(r"DRAG calibration out of spec for Q\d+ with N=(\d+)", w))
            ),
            default=float("nan"),
        )
        for _, start, _, oi in iter_files(OUTPUT_INFO, routines=["calibrate_1q"])
    }
).sort_index()

# Delays per run against the lowest DRAG N warned
pd.crosstab(
    delays.reindex(drag_n.index).rename("FineFrequency delays"),
    drag_n.fillna(0).astype(int).rename("lowest DRAG N warned (0 = none)"),
)

69bc440 | author 2025-02-17 16:08:24 +0000 | committer 2025-03-28 17:16:21 +0000 | Adjust drag_limit, drag Ns, and fine_frequency sensitivity
48b8810 | author 2025-04-17 11:22:37 +0100 | committer 2025-04-17 11:24:10 +0100 | Add fit_quality_threshold arg for run_benchmarking, default to 0.9 rather than 0.8
-    drag_limit = 4e-9
+    drag_limit = 9e-9
-        for delay in [0.1e-6, 0.25e-6]:
+        for delay in [0.1e-6, 0.25e-6, 0.5e-6]:
-        for N in [1, 2, 4]:
+        for N in [2, 3, 4]:


lowest DRAG N warned (0 = none),0,1,2,3,4
FineFrequency delays,,,,,
1.0,0,0,1,0,0
2.0,205,29,5,0,5
2.5,1,0,1,0,0
3.0,96,0,774,43,86


69bc440 was written on 2025-02-17 16:08:24 (author date), minutes before the first 3-delay run, but only committed on 2025-03-28 17:16:21, the evening the new code came back for good. $N=1$ DRAG warnings turn up in 29 runs with 2 delays and in none with 3, so the ladder and the delays changed together.

Commit 48b8810 (2025-04-17) raises the benchmarking `fit_quality_threshold` default from 0.8 to 0.9. No era 1 good value sits below 0.9 ([datasheet 3, section 6](../../docs/datasheets/03_benchmarking_files.md#the-shared-gate)), so this isn't a break.

## 5. Both 2025 outages end with new values written outside the routines

A3 found DS2 changes that fall on no tracked run's `snap_after`. For each snapshot $t$ with at least 50 of them, we look back over the earlier snapshots $s$ within 120 days and ask how many of the new values already equalled the value in force at $s$. The best match is

$$\text{share\_equal}(t) = \max_{s}\ \frac{1}{n_t}\sum_{k=1}^{n_t} \mathbb{1}\big[v_k(t) = v_k(s)\big],$$

where $v_k(t)$ is the new value of change $k$ and $n_t$ the number of untracked changes at $t$. A share near 1 means an earlier hardware model was put back (a rollback). A share near 0 means nothing in the last 120 days matches (unmatched): either new values, or a restore from further back.

We also look at the smaller snapshots, the first untracked snapshot and the first run after each outage, and the last DS2 DC-bias change.

In [9]:
from qpu_drift.derived.run_map import differs
from qpu_drift.derived.run_map.clean import ds2_series
from qpu_drift.derived.run_map.constants import DS2_TEMPLATES
from qpu_drift.sources.hardware_models import load_values

KEY = ["routine", "param", "entity"]
values = load_values()
series = ds2_series(values[values["path_template"].isin(DS2_TEMPLATES)]).sort_values(
    [*KEY, "valid_from_ts"]
)

# Drop unchanged halves of complex params
series = series[
    differs(series["value"], series.groupby(KEY)["value"].shift()).fillna(True).to_numpy(bool)
]

# Changes that land on no tracked run's snap_after
after = runs.dropna(subset=["snap_after"]).groupby("routine")["snap_after"].apply(set)
chg = series[series["valid_from_ts"] > series["valid_from_ts"].min()]
untracked = chg[
    [t not in after.get(r, set()) for r, t in zip(chg["routine"], chg["valid_from_ts"])]
]
print(
    f"changes no tracked run made: {len(untracked):,} "
    f"on {untracked['valid_from_ts'].nunique()} snapshots"
)

# Snapshots with 50 changes or more: the earlier snapshot (within 120 days)
# whose values the new ones most often equal
snaps = pd.DatetimeIndex(pd.to_datetime(ds2["timestamp"])).unique().sort_values().as_unit("ns")
state = (
    series[[*KEY, "valid_from_ts", "value"]]
    .rename(columns={"valid_from_ts": "t", "value": "old"})
    .sort_values("t")
)
rows = []
for t, grp in untracked.groupby("valid_from_ts"):
    if len(grp) < 50:
        continue
    earlier = snaps[(snaps < t) & (snaps >= t - pd.Timedelta("120D"))]
    left = grp[[*KEY, "value"]].merge(pd.DataFrame({"t": earlier}), how="cross").sort_values("t")
    m = pd.merge_asof(left, state, on="t", by=KEY)  # each key's value as of each earlier snapshot
    share = (
        pd.Series(~differs(m["value"], m["old"]).fillna(True).to_numpy(bool))
        .groupby(m["t"].to_numpy())
        .mean()
    )
    rows.append(
        {
            "snapshot": t,
            "changes": len(grp),
            "routines": ", ".join(sorted(grp["routine"].unique())),
            "best_earlier": share.idxmax(),
            "share_equal": round(share.max(), 2),
        }
    )
restores = pd.DataFrame(rows)
display(restores)

# Snapshots under 50 changes
per_snap = untracked.groupby("valid_from_ts").size()
small = per_snap[per_snap < 50]
print(f"snapshots under 50 changes: {len(small)}, {small.sum()} changes, largest {small.max()}")
display(
    untracked[untracked["valid_from_ts"].isin(small.index)]
    .groupby(["routine", "param"])
    .size()
    .rename("changes")
)

# Next untracked snapshot, next rollback and first non-empty run after each outage
rollbacks = restores.loc[restores["share_equal"] >= 0.9, "snapshot"]
nonempty = runs[runs["n_result_ids"] > 0]
for _, g in gaps[(gaps["size"] >= 3) & (gaps["min"] >= pd.Timestamp("2025-01-01"))].iterrows():
    end = g["max"] + pd.Timedelta("1D")
    print(
        f"outage {g['min'].date()} to {g['max'].date()}: "
        f"next untracked snapshot {untracked.loc[untracked['valid_from_ts'] >= end, 'valid_from_ts'].min()}, "
        f"next rollback {rollbacks[rollbacks >= end].min()}, "
        f"first run back {nonempty.loc[nonempty['start'] >= end, 'start'].min()}"
    )

# Last DS2 DC-bias change
bias_last = series.loc[series["param"].str.startswith("dc_bias"), "valid_from_ts"].max()
print("last DS2 DC-bias change:", bias_last)

changes no tracked run made: 2,543 on 30 snapshots


,snapshot,changes,routines,best_earlier,share_equal
0,2025-02-12 13:40:14,64,calibrate_1q,2025-01-22 08:16:49,0.02
1,2025-02-12 17:34:10,151,"calibrate_1q, calibrate_2q",2025-01-01 08:00:13,0.00
2,2025-02-23 08:00:14,155,calibrate_2q,2025-01-01 08:00:13,0.00
3,2025-03-18 11:42:44,161,mixer_calibration,2025-02-15 19:42:00,0.12
4,2025-03-18 19:06:53,256,"calibrate_1q, calibrate_2q",2025-01-01 08:00:13,0.00
5,2025-05-12 09:15:09,105,calibrate_2q,2025-01-12 19:05:13,0.00
6,2025-09-03 19:05:16,246,"calibrate_1q, calibrate_2q",2025-09-02 20:13:31,0.98
7,2026-02-26 19:05:15,246,"calibrate_1q, calibrate_2q",2026-02-25 21:09:14,1.00
8,2026-03-24 12:03:12,490,"calibrate_1q, calibrate_2q, mixer_calibration",2026-03-05 20:11:50,1.00
9,2026-05-20 20:13:10,67,mixer_calibration,2026-04-14 19:42:58,0.21


snapshots under 50 changes: 18, 232 changes, largest 36


routine       param          
calibrate_1q  amp                14
              drag               12
              freq               45
calibrate_2q  amp                33
              cancel_scale_im    33
              cancel_scale_re    33
              cr_scale_im        31
              cr_scale_re        31
Name: changes, dtype: int64

outage 2025-02-05 to 2025-02-11: next untracked snapshot 2025-02-12 13:40:14, next rollback 2025-09-03 19:05:16, first run back 2025-02-12 13:41:10
outage 2025-03-10 to 2025-03-17: next untracked snapshot 2025-03-18 11:42:44, next rollback 2025-09-03 19:05:16, first run back 2025-03-18 19:06:59
outage 2026-03-12 to 2026-03-23: next untracked snapshot 2026-03-24 12:03:12, next rollback 2026-03-24 12:03:12, first run back 2026-03-24 12:46:35
last DS2 DC-bias change: 2026-05-25 22:01:25


2,543 changes on 30 snapshots don't come from a tracked run. The 12 snapshots with 50 or more split cleanly in two. Four are rollbacks (share 0.98 to 1.00): 2025-09-03 19:05:16 (back to 2025-09-02 20:13:31), 2026-02-26 19:05:15 (to 2026-02-25 21:09:14), 2026-03-24 12:03:12 (to 2026-03-05 20:11:50) and 2026-07-05 21:25:30 (to 2026-07-01 20:04:47). The other eight are unmatched (share 0.00 to 0.23): 2025-02-12 13:40:14 and 17:34:10, 2025-02-23, 2025-03-18 11:42:44 and 19:06:53, 2025-05-12, and the mixer on 2026-05-20 and 05-21. For the 2025-02 and 2025-03 ones the 120 days reach back past the first DS2 snapshot (2025-01-01), so the lookback is shorter.

The 18 smaller snapshots hold 232 changes (36 at most): for `calibrate_1q` freq 45, amp 14 and drag 12, and for `calibrate_2q` 31 to 33 each of amp, `cancel_scale` and `cr_scale`. DS2's last DC-bias change is at 2026-05-25 22:01:25.

The outages end in two different ways. After each 2025 outage the first untracked snapshot is unmatched (2025-02-12 13:40:14, share 0.02; 2025-03-18 11:42:44, share 0.12), and the first run back starts within a minute of an unmatched write (2025-02-12 13:41:10 and 2025-03-18 19:06:59). After the 2026-03 outage the first untracked snapshot is a rollback to values from before the outage (2026-03-24 12:03:12), and the first run follows at 12:46:35.

**Downstream.** An untracked write on its own isn't a break. A rollback puts back an earlier state, and the other unmatched writes come in the middle of normal running. We drop any DS2 difference between consecutive runs that spans one (`periods.UNTRACKED_DS2` with `periods.spans()`), and for the smaller snapshots we compare each run's `before` with the previous run's `after` per entity.

The end of each 2025 outage is different. Every routine stopped for a week, there's no fridge record before 2025-08 (section 1), and the machine restarted on values no tracked run wrote. So we can't assume the state before and after is the same, and we put a break at each outage end (2025-02-12 and 2025-03-18). The 2026-03 outage already ends in breaks: the thermal-cycle end from DS4, and the coupling flip on 2026-03-24 (section 3).

## 6. The register in `periods.py` matches the data

Last, we compare every entry in `qpu_drift.periods` with what we found above; `match` is True when they agree. A study period runs from the latest first day of its sources to the day after the earliest last day,

$$\Big[\max_{s}\, t_s^{\text{first}},\ \ \min_{s}\, t_s^{\text{last}} + 1\,\text{d}\Big),$$

except DS1 and DS3, which starts at the first benchmarking run linked to DS3, since that link goes through the run map. Outages are the section 2 gaps from 2025. Breaks match on the date for day-level entries and to the second for timed ones, and `UNTRACKED_DS2` matches the section 5 table (a rollback when $\text{share\_equal} \ge 0.9$). A computed time with no break on it shows up at the end of the break table.

In [10]:
from itertools import zip_longest

from qpu_drift import periods

# First and last time of each source
day = pd.Timedelta("1D")
first = {
    "DS1": ds1["time_start"].min(),
    "DS2": ds2["timestamp"].min(),
    "DS3": ds3["created_at"].min(),
    "DS4": ds4.index.min().tz_localize(None),
}
last = {
    "DS1": ds1["time_start"].max(),
    "DS2": ds2["timestamp"].max(),
    "DS3": ds3["created_at"].max(),
    "DS4": ds4.index.max().tz_localize(None),
}


def span(sources, start=None):
    """[first day, day after the last day] where all the sources have data."""
    start = start if start is not None else max(first[s] for s in sources)
    return start.normalize(), min(last[s] for s in sources).normalize() + day


# Computed study periods, outages and DS4 gaps
computed = {
    "all": span(["DS1", "DS2", "DS3", "DS4"]),
    "ds1_ds2_ds3": span(["DS1", "DS2", "DS3"]),
    "ds1_ds3": span(["DS1", "DS3"], start=runs.loc[runs["ds3_created_at"].notna(), "start"].min()),
    "ds3": span(["DS3"]),
}
outages = [
    (g["min"], g["max"] + day)
    for _, g in gaps[(gaps["size"] >= 3) & (gaps["min"] >= pd.Timestamp("2025-01-01"))].iterrows()
]
big_steps = step[step > pd.Timedelta("6h")]
ds4_gaps = [((t - s).tz_localize(None), t.tz_localize(None)) for t, s in big_steps.items()]
rows = [("study period " + k, periods.STUDY_PERIODS.get(k), computed[k]) for k in computed]
rows += [("outage", a, b) for a, b in zip_longest(periods.OUTAGES, outages)]
rows += [("DS4 gap", a, b) for a, b in zip_longest(periods.DS4_GAPS, ds4_gaps)]
display(
    pd.DataFrame(rows, columns=["entry", "register", "computed"]).assign(
        match=lambda d: d["register"] == d["computed"]
    )
)

# Computed times for each kind of break in the register
ff_seg = delays.groupby(segment).agg(delays="first", first=lambda s: s.index.min(), runs="size")
long = ff_seg[ff_seg["runs"] >= 10]
gating = ds3.groupby("gating_regime")["created_at"].min()
evidence = {
    "couplings": list(change["created_at"].iloc[1:]),
    "DS1 restore": [first["DS1"]],
    "outage ends": [end for _, end in outages],
    "code": list(long.loc[long["delays"] != long["delays"].shift(), "first"].iloc[1:]),
    "gating era 1": [gating["era1_filter"]],
    "gating era 2": [gating["era2_ungated"]],
    "gating era 3": [gating["era3_postproc"]],
    "openpulse": [ds3.loc[ds3["has_openpulse"], "created_at"].min()],
    "DC bias": [bias_last],
}


def same(at: pd.Timestamp, e: pd.Timestamp) -> bool:
    """Day-level register entries match on the date, timed ones to the second."""
    return e.normalize() == at if at == at.normalize() else e.floor("s") == at


# Match each register break to a computed time, and list computed times no break sits on
rows = []
for b in periods.BREAKS:
    if b.source == "ds4":  # thermal-cycle ends come from DS4, not checked here
        continue
    kind = next(k for k in evidence if k in b.what)
    hit = next((e for e in evidence[kind] if same(b.at, e)), None)
    rows.append((b.at, b.what, hit, hit is not None))
rows += [
    (None, f"not in register: {k}", e, False)
    for k, es in evidence.items()
    for e in es
    if not any(same(b.at, e) for b in periods.BREAKS)
]
display(pd.DataFrame(rows, columns=["register", "what", "computed", "match"]))

# Untracked snapshots from section 5: kind, restored snapshot and routines
computed_untracked = {
    r["snapshot"]: (
        "rollback" if r["share_equal"] >= 0.9 else "unmatched",
        r["best_earlier"] if r["share_equal"] >= 0.9 else None,
        tuple(r["routines"].split(", ")),
    )
    for _, r in restores.iterrows()
}
rows = [
    (u.at, (u.kind, u.restores, u.routines), computed_untracked.get(u.at))
    for u in periods.UNTRACKED_DS2
]
rows += [
    (t, None, c)
    for t, c in computed_untracked.items()
    if t not in {u.at for u in periods.UNTRACKED_DS2}
]
pd.DataFrame(rows, columns=["snapshot", "register", "computed"]).assign(
    match=lambda d: d["register"] == d["computed"]
)

,entry,register,computed,match
0,study period all,"(2025-08-01 00:00:00, 2026-08-19 00:00:00)","(2025-08-01 00:00:00, 2026-08-19 00:00:00)",True
1,study period ds1_ds2_ds3,"(2025-01-01 00:00:00, 2026-08-19 00:00:00)","(2025-01-01 00:00:00, 2026-08-19 00:00:00)",True
2,study period ds1_ds3,"(2024-10-15 00:00:00, 2026-08-19 00:00:00)","(2024-10-15 00:00:00, 2026-08-19 00:00:00)",True
3,study period ds3,"(2024-05-24 00:00:00, 2026-09-15 00:00:00)","(2024-05-24 00:00:00, 2026-09-15 00:00:00)",True
4,outage,"(2025-02-05 00:00:00, 2025-02-12 00:00:00)","(2025-02-05 00:00:00, 2025-02-12 00:00:00)",True
5,outage,"(2025-03-10 00:00:00, 2025-03-18 00:00:00)","(2025-03-10 00:00:00, 2025-03-18 00:00:00)",True
6,outage,"(2026-03-12 00:00:00, 2026-03-24 00:00:00)","(2026-03-12 00:00:00, 2026-03-24 00:00:00)",True
7,DS4 gap,"(2026-03-17 00:00:00, 2026-03-17 18:00:00)","(2026-03-17 00:00:00, 2026-03-17 18:00:00)",True
8,DS4 gap,"(2026-06-11 18:00:00, 2026-07-02 18:00:00)","(2026-06-11 18:00:00, 2026-07-02 18:00:00)",True
9,DS4 gap,"(2026-07-07 06:00:00, 2026-07-08 18:00:00)","(2026-07-07 06:00:00, 2026-07-08 18:00:00)",True


,register,what,computed,match
0,2024-05-28 00:00:00,couplings 2-1 and 25-17 reverse,2024-05-28 20:10:56.372690,True
1,2024-09-18 00:00:00,DS1 restore,2024-09-18 11:53:21.161967,True
2,2025-02-12 00:00:00,outage ends (hand-written DS2 values),2025-02-12 00:00:00.000000,True
3,2025-02-17 16:11:02,calibrate_1q code change,2025-02-17 16:11:02.000000,True
4,2025-03-18 00:00:00,outage ends (hand-written DS2 values),2025-03-18 00:00:00.000000,True
5,2025-03-20 12:00:58,calibrate_1q old code back,2025-03-20 12:00:58.000000,True
6,2025-03-28 19:44:15,calibrate_1q new code back,2025-03-28 19:44:15.000000,True
7,2025-07-07 00:00:00,"DS3 gating era 2, error fields start",2025-07-07 23:10:10.082632,True
8,2025-08-22 00:00:00,DS3 gating era 3,2025-08-22 23:12:56.049756,True
9,2026-03-24 00:00:00,couplings 1-10 and 1-2 reverse,2026-03-24 23:48:40.391568,True


,snapshot,register,computed,match
0,2025-02-12 13:40:14,"(unmatched, None, (calibrate_1q,))","(unmatched, None, (calibrate_1q,))",True
1,2025-02-12 17:34:10,"(unmatched, None, (calibrate_1q, calibrate_2q))","(unmatched, None, (calibrate_1q, calibrate_2q))",True
2,2025-02-23 08:00:14,"(unmatched, None, (calibrate_2q,))","(unmatched, None, (calibrate_2q,))",True
3,2025-03-18 11:42:44,"(unmatched, None, (mixer_calibration,))","(unmatched, None, (mixer_calibration,))",True
4,2025-03-18 19:06:53,"(unmatched, None, (calibrate_1q, calibrate_2q))","(unmatched, None, (calibrate_1q, calibrate_2q))",True
5,2025-05-12 09:15:09,"(unmatched, None, (calibrate_2q,))","(unmatched, None, (calibrate_2q,))",True
6,2025-09-03 19:05:16,"(rollback, 2025-09-02 20:13:31, (calibrate_1q,...","(rollback, 2025-09-02 20:13:31, (calibrate_1q,...",True
7,2026-02-26 19:05:15,"(rollback, 2026-02-25 21:09:14, (calibrate_1q,...","(rollback, 2026-02-25 21:09:14, (calibrate_1q,...",True
8,2026-03-24 12:03:12,"(rollback, 2026-03-05 20:11:50, (calibrate_1q,...","(rollback, 2026-03-05 20:11:50, (calibrate_1q,...",True
9,2026-05-20 20:13:10,"(unmatched, None, (mixer_calibration,))","(unmatched, None, (mixer_calibration,))",True


All 4 study periods, 3 outages, 3 DS4 gaps, 12 breaks and 12 untracked snapshots match. The 2026-03 outage ends on 2026-03-24, the day of the coupling flip, so it needs no break of its own. The one computed time with no break is the start of gating era 1 (2024-10-15 19:39:16).

**Downstream.** We leave era 1 out of `BREAKS` on purpose. Era 0 has no $R^2$ anyway (section 3), and only the DS3-alone study period reaches back before 2024-10-15, so only DS3-only work crosses it: split at `gating_regime` there. This check doesn't cover the DS4 break (thermal-cycle end), `THERMAL_CYCLES`, `SETTLING`, `UNTRACKED_DS2_OTHER`, `F01_FLAGS` or `INCIDENTS`.

## Conclusion

| Result | Status |
|---|---|
| All four sources overlap from 2025-08-01 to 2026-08-18, and DS1 to DS3 from 2025-01-01 | Confirmed |
| Three outages stop every routine: 2025-02-05 to 02-11, 2025-03-10 to 03-17 and 2026-03-12 to 03-23. DS1 and DS3 have no rows inside them, DS2 only a few | Confirmed |
| DS3 has gaps of its own, the longest 2024-08-18 to 10-14 (58 days) | Confirmed |
| `calibrate_1q` ran the new code from 2025-02-17 16:11, the old code again from 2025-03-20 12:00 to 03-28 08:45, then the new code from 2025-03-28 19:44 | Confirmed |
| Couplings flip on 2024-05-28 and 2026-03-24, DS3 gating eras start on 2024-10-15, 2025-07-07 and 2025-08-22, and `openpulse_features` on 2026-05-21 | Confirmed |
| Untracked DS2 snapshots are rollbacks or unmatched. On their own they aren't breaks: we drop DS2 differences that span one | Confirmed |
| Both 2025 outages end with unmatched DS2 writes and no fridge record, so each outage end is a break. The 2026-03 outage ends in a rollback and already has breaks | Confirmed |
| DS2's last DC-bias change is at 2026-05-25 22:01:25 | Confirmed |
| Reduced `calibrate_1q` runs aren't breaks; the `measured` flag handles them | Confirmed |
| `periods.py` matches everything computed here; the era 1 start is left out of `BREAKS` on purpose | Confirmed |